In [ ]:
import time
import pandas as pd
from tqdm import tqdm
!pip install neo4j
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer, util

#model = SentenceTransformer('all-MiniLM-L6-v2')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 4.5 MB/s eta 0:00:00


In [ ]:
#----CONFIG-------------
NEO4J_URI      = ""
NEO4J_USER     = ""
NEO4J_PASSWORD = ""
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
EMBEDDING_DIMS  = 384
BATCH_SIZE      = 128
NEO4J_BATCH     = 100

In [ ]:
class EmbeddingUploader:
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.driver.verify_connectivity()
        print("✅ Connected to Neo4j Aura")

    def close(self):
        self.driver.close()

    def run(self, query: str, params: dict = {}):
        with self.driver.session() as session:
            session.run(query, params).consume()

    def query(self, query_str: str, params: dict = {}) -> list:
        """Run a read query — fetches all records inside the session."""
        with self.driver.session() as session:
            result = session.run(query_str, params)
            return [r.data() for r in result]

    # ── CREATE VECTOR INDEX ───────────────────────────────────────────────────

    def create_vector_index(self, dims: int):
        """
        Create a Neo4j vector index on Product.embedding
        Neo4j Aura supports vector indexes from version 5.11+
        """
        print(f"\n📐 Creating vector index (dims={dims})...")

        # Drop if exists so we can recreate cleanly
        try:
            self.run("DROP INDEX product_embedding_index IF EXISTS")
        except Exception:
            pass

        self.run(f"""
            CREATE VECTOR INDEX product_embedding_index IF NOT EXISTS
            FOR (p:Product)
            ON p.embedding
            OPTIONS {{
                indexConfig: {{
                    `vector.dimensions`:   {dims},
                    `vector.similarity_function`: 'cosine'
                }}
            }}
        """)
        # Aura needs a moment to build the index
        time.sleep(3)
        print("   ✅ Vector index created: product_embedding_index")

    # ── FETCH PRODUCTS ────────────────────────────────────────────────────────

    def fetch_products(self) -> pd.DataFrame:
        """Fetch all products with their text fields from Neo4j."""
        print("\n📥 Fetching products from Neo4j...")
        rows = self.query("""
            MATCH (p:Product)
            OPTIONAL MATCH (p)-[:HAS_FEATURE]->(f:Feature)
            OPTIONAL MATCH (p)-[:BELONGS_TO]->(c:Category)
            OPTIONAL MATCH (p)-[:MADE_BY]->(b:Brand)
            RETURN
                p.asin           AS asin,
                p.title          AS title,
                p.average_rating AS avg_rating,
                p.price          AS price,
                collect(DISTINCT f.name) AS features,
                collect(DISTINCT c.name) AS categories,
                b.name           AS brand
        """)
        df   = pd.DataFrame(rows)
        print(f"   ✅ Fetched {len(df):,} products")
        return df

    # ── WRITE EMBEDDINGS BACK ─────────────────────────────────────────────────

    def store_embeddings(self, asin_embeddings: list[dict]):
        """Write embedding vectors back onto Product nodes in batches."""
        total = len(asin_embeddings)
        with tqdm(total=total, desc="   ⬆ Storing embeddings", unit="nodes") as pbar:
            for i in range(0, total, NEO4J_BATCH):
                batch = asin_embeddings[i : i + NEO4J_BATCH]
                self.run(
                    """
                    UNWIND $rows AS row
                    MATCH (p:Product {asin: row.asin})
                    CALL db.create.setNodeVectorProperty(p, 'embedding', row.embedding)
                    """,
                    {"rows": batch},
                )
                pbar.update(len(batch))
        print(f"   ✅ Stored {total:,} embeddings in Neo4j")

    # ── VERIFY ────────────────────────────────────────────────────────────────

    def verify(self):
        print("\n🔍 Verifying embeddings...")
        rows = self.query("""
            MATCH (p:Product)
            WHERE p.embedding IS NOT NULL
            RETURN count(p) AS embedded,
                   size(p.embedding) AS dims
            LIMIT 1
        """)
        if rows:
            r = rows[0]
            print(f"   ✅ {r['embedded']:,} products have embeddings (dims={r['dims']})")
        else:
            print("   ⚠️  No embeddings found — something may have gone wrong")

    def sample_similarity_search(self, model: SentenceTransformer, query: str = "wireless noise cancelling headphones"):
        """Run a quick test similarity search to confirm everything works."""
        print(f"\n🔎 Test search: '{query}'")
        query_embedding = model.encode(query).tolist()
        rows = self.query(
            """
            CALL db.index.vector.queryNodes(
                'product_embedding_index', 5, $embedding
            )
            YIELD node AS p, score
            RETURN p.title AS title, p.average_rating AS rating, score
            ORDER BY score DESC
            """,
            {"embedding": query_embedding},
        )
        if rows:
            print("   Top 5 results:")
            for i, r in enumerate(rows, 1):
                print(f"   {i}. [{r['score']:.3f}] {str(r['title'])[:80]}  ⭐{r['rating']}")
        else:
            print("   ⚠️  No results — index may still be building, wait a few seconds and retry")


# ─── TEXT BUILDER ─────────────────────────────────────────────────────────────

def build_text_for_embedding(row: dict) -> str:
    """
    Combine product fields into a single rich text string for embedding.
    More context = better semantic search quality.

    Format:
        Title: <title>
        Brand: <brand>
        Category: <cat1>, <cat2>
        Features: <feat1>, <feat2>, ...
        Rating: <x>/5
    """
    parts = []

    title = str(row.get("title") or "").strip()
    if title:
        parts.append(f"Title: {title}")

    brand = str(row.get("brand") or "").strip()
    if brand and brand.lower() not in ("none", "nan", ""):
        parts.append(f"Brand: {brand}")

    categories = row.get("categories") or []
    if categories:
        cats = ", ".join(str(c) for c in categories if c)
        if cats:
            parts.append(f"Category: {cats}")

    features = row.get("features") or []
    if features:
        feats = ", ".join(str(f) for f in features[:10] if f)  # cap at 10
        if feats:
            parts.append(f"Features: {feats}")

    rating = row.get("avg_rating")
    if rating:
        parts.append(f"Rating: {rating}/5")

    return "\n".join(parts)


# ─── MAIN PIPELINE ────────────────────────────────────────────────────────────

def run_embeddings(
    uri:      str = NEO4J_URI,
    user:     str = NEO4J_USER,
    password: str = NEO4J_PASSWORD,
):
    # 1. Load model
    print(f"\n🤖 Loading embedding model: {EMBEDDING_MODEL}")
    model = SentenceTransformer(EMBEDDING_MODEL)
    print(f"   ✅ Model loaded — output dims: {model.get_sentence_embedding_dimension()}")

    uploader = EmbeddingUploader(uri, user, password)

    try:
        # 2. Create vector index in Neo4j
        uploader.create_vector_index(dims=EMBEDDING_DIMS)

        # 3. Fetch products from Neo4j
        products_df = uploader.fetch_products()

        if products_df.empty:
            print("❌ No products found in Neo4j. Run upload_to_neo4j.py first.")
            return

        # 4. Build text strings for each product
        print("\n📝 Building text for embedding...")
        products_df["text"] = products_df.apply(build_text_for_embedding, axis=1)

        # Quick preview
        print("\n   Sample text for first product:")
        print("   " + products_df["text"].iloc[0].replace("\n", "\n   "))

        # 5. Generate embeddings in batches
        print(f"\n⚙️  Generating embeddings in batches of {BATCH_SIZE}...")
        all_texts = products_df["text"].tolist()
        all_asins = products_df["asin"].tolist()

        asin_embeddings = []
        for i in tqdm(range(0, len(all_texts), BATCH_SIZE), desc="   🔢 Embedding batches"):
            batch_texts = all_texts[i : i + BATCH_SIZE]
            batch_asins = all_asins[i : i + BATCH_SIZE]

            # encode() returns a numpy array — convert to plain Python list for Neo4j
            vectors = model.encode(
                batch_texts,
                show_progress_bar=False,
                normalize_embeddings=True,   # cosine sim works best with normalized vecs
            )

            for asin, vec in zip(batch_asins, vectors):
                asin_embeddings.append({
                    "asin":      asin,
                    "embedding": vec.tolist(),
                })

        print(f"   ✅ Generated {len(asin_embeddings):,} embeddings")

        # 6. Write embeddings back to Neo4j
        print("\n💾 Writing embeddings to Neo4j...")
        uploader.store_embeddings(asin_embeddings)

        # 7. Verify + test search
        uploader.verify()
        uploader.sample_similarity_search(model)

        print("\n✅ Embeddings complete! Neo4j is now ready for semantic search.")
        print("   Next step: build the RAG query layer (Step 5)")

    except Exception as e:
        print(f"\n❌ Failed: {e}")
        raise

    finally:
        uploader.close()


# ─── STANDALONE ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import sys
    if "<your-instance>" in NEO4J_URI:
        print(
            "❌  Set your credentials at the top of the file,\n"
            "    or call run_embeddings(uri=..., user=..., password=...) from your notebook.\n"
        )
        sys.exit(1)
    run_embeddings()


🤖 Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ✅ Model loaded — output dims: 384
✅ Connected to Neo4j Aura

📐 Creating vector index (dims=384)...
   ✅ Vector index created: product_embedding_index

📥 Fetching products from Neo4j...
   ✅ Fetched 5,027 products

📝 Building text for embedding...

   Sample text for first product:
   Title: FS-1051 FATSHARK TELEPORTER V3 HEADSET
   Brand: Fat Shark
   Category: ElectronicsTelevision & VideoVideo Glasses
   Rating: 3.5/5

⚙️  Generating embeddings in batches of 128...


   🔢 Embedding batches: 100%|██████████| 40/40 [06:33<00:00,  9.83s/it]


   ✅ Generated 5,027 embeddings

💾 Writing embeddings to Neo4j...


   ⬆ Storing embeddings: 100%|██████████| 5027/5027 [00:25<00:00, 196.38nodes/s]


   ✅ Stored 5,027 embeddings in Neo4j

🔍 Verifying embeddings...
   ✅ 5,000 products have embeddings (dims=384)

🔎 Test search: 'wireless noise cancelling headphones'
   Top 5 results:
   1. [0.805] Beats Studio3 Wireless Noise Cancelling Over-Ear Headphones - Apple W1 Headphone  ⭐4.7
   2. [0.800] Bluetooth Earbuds Wireless Active Noise Canceling Headphones in-Ear Earphones wi  ⭐3.2
   3. [0.787] iLive Noise Canceling Wireless Headphones  ⭐3.9
   4. [0.786] GIEC Wireless Earbuds Bluetooth Earbuds Noise Cancelling Earbuds with Mic True W  ⭐4.0
   5. [0.786] Sony WH-XB900N Wireless Noise Canceling Extra Bass Headphones, Gray  ⭐4.3

✅ Embeddings complete! Neo4j is now ready for semantic search.
   Next step: build the RAG query layer (Step 5)


In [ ]:
run_embeddings(uri=NEO4J_URI,user=NEO4J_USER,password=NEO4J_PASSWORD)